In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

dbutils.widgets.removeAll()

dbutils.widgets.text("catalogo", "catalog_dev")
dbutils.widgets.text("esquema_source", "silver")
dbutils.widgets.text("tabla_source", "ecommerce_sales_prediction_silver")
dbutils.widgets.text("esquema_sink", "gold")

dbutils.widgets.text("fact_table", "fact_sales")
dbutils.widgets.text("agg_category_table", "agg_sales_by_category")
dbutils.widgets.text("agg_segment_table", "agg_sales_by_segment")
dbutils.widgets.text("agg_time_table", "agg_sales_by_month")
dbutils.widgets.text("kpi_table", "kpi_overview")

catalogo = dbutils.widgets.get("catalogo")
esquema_source  = dbutils.widgets.get("esquema_source")
tabla_source  = dbutils.widgets.get("tabla_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

fact_table = dbutils.widgets.get("fact_table")
agg_category_table = dbutils.widgets.get("agg_category_table")
agg_segment_table = dbutils.widgets.get("agg_segment_table")
agg_time_table = dbutils.widgets.get("agg_time_table")
kpi_table = dbutils.widgets.get("kpi_table")

df_silver = spark.table(f"{catalogo}.{esquema_source}.{tabla_source}")

In [0]:
fact_df = df_silver.select(
    "Date",
    "Year",
    "Month",
    "Quarter",
    "Product_Category",
    "Customer_Segment",
    "Units_Sold",
    "Price",
    "Discount",
    "Net_Unit_Price",
    "Revenue",
    "Discount_Rate",
    "Discount_Level"
)

fact_df.write.mode("overwrite") \
    .saveAsTable(f"{catalogo}.{esquema_sink}.{fact_table}")

In [0]:
df_silver.groupBy("Customer_Segment").agg(
    sum("Revenue").alias("Total_Revenue"),
    avg("Revenue").alias("Avg_Revenue_Per_Transaction"),
    sum("Units_Sold").alias("Total_Units")
).write.mode("overwrite") \
 .saveAsTable(f"{catalogo}.{esquema_sink}.{agg_segment_table}")

In [0]:
df_silver.groupBy("Year","Month").agg(
    sum("Revenue").alias("Monthly_Revenue"),
    sum("Units_Sold").alias("Monthly_Units"),
    avg("Discount_Rate").alias("Monthly_Avg_Discount")
).orderBy("Year","Month") \
 .write.mode("overwrite") \
 .saveAsTable(f"{catalogo}.{esquema_sink}.{agg_time_table}")

In [0]:
kpi_df = df_silver.agg(
    sum("Revenue").alias("Total_Revenue"),
    sum("Units_Sold").alias("Total_Units_Sold"),
    avg("Discount_Rate").alias("Avg_Discount_Rate"),
    avg("Net_Unit_Price").alias("Avg_Net_Unit_Price")
)

kpi_df.write.mode("overwrite") \
    .saveAsTable(f"{catalogo}.{esquema_sink}.{kpi_table}")